# Frontend / Load Testing Assignment

**Task:** Use tools like JMeter or Gatling to perform load testing on a web application and
analyze the results to identify performance bottlenecks.

This notebook sets up **both** a JMeter test plan and a Gatling simulation against a simple
target web app, then walks through how to read the results and spot bottlenecks.

## 1. Target Web Application
We'll load-test a minimal Flask endpoint with an artificial delay so bottlenecks are visible.

In [1]:
import os
os.makedirs('project', exist_ok=True)
with open('project/app.py', 'w') as f:
    f.write('''from flask import Flask, jsonify
import time, random

app = Flask(__name__)

@app.route("/api/products")
def products():
    # simulate variable DB latency -> a bottleneck to find during load testing
    time.sleep(random.uniform(0.05, 0.4))
    return jsonify([{"id": i, "name": f"Product {i}"} for i in range(10)])

if __name__ == "__main__":
    app.run(port=5000)
''')
print("Created project/app.py — run with: python project/app.py")


Created project/app.py — run with: python project/app.py


## 2. JMeter Test Plan (`.jmx`)
This is a ready-to-open JMeter test plan: 50 users, 10 loops, hitting `/api/products`.

In [2]:
import os
os.makedirs('project/jmeter', exist_ok=True)
jmx = '''<?xml version="1.0" encoding="UTF-8"?>
<jmeterTestPlan version="1.2" properties="5.0">
  <hashTree>
    <TestPlan testname="LoadTestPlan" enabled="true">
      <hashTree>
        <ThreadGroup testname="ProductsLoad" enabled="true">
          <intProp name="ThreadGroup.num_threads">50</intProp>
          <intProp name="ThreadGroup.ramp_time">10</intProp>
          <longProp name="ThreadGroup.duration">60</longProp>
          <stringProp name="LoopController.loops">10</stringProp>
          <hashTree>
            <HTTPSamplerProxy testname="GET /api/products" enabled="true">
              <stringProp name="HTTPSampler.domain">localhost</stringProp>
              <stringProp name="HTTPSampler.port">5000</stringProp>
              <stringProp name="HTTPSampler.path">/api/products</stringProp>
              <stringProp name="HTTPSampler.method">GET</stringProp>
            </HTTPSamplerProxy>
            <hashTree/>
          </hashTree>
        </ThreadGroup>
      </hashTree>
    </TestPlan>
  </hashTree>
</jmeterTestPlan>
'''
with open('project/jmeter/load_test.jmx', 'w') as f:
    f.write(jmx)
print("Created project/jmeter/load_test.jmx")
print("Run headless: jmeter -n -t project/jmeter/load_test.jmx -l results.jtl -e -o report/")


Created project/jmeter/load_test.jmx
Run headless: jmeter -n -t project/jmeter/load_test.jmx -l results.jtl -e -o report/


## 3. Gatling Simulation (Scala)

In [3]:
import os
os.makedirs('project/gatling/src/main/scala', exist_ok=True)
scala_sim = '''import io.gatling.core.Predef._
import io.gatling.http.Predef._
import scala.concurrent.duration._

class ProductsLoadSimulation extends Simulation {

  val httpProtocol = http.baseUrl("http://localhost:5000")

  val scn = scenario("Load products endpoint")
    .exec(http("get_products").get("/api/products"))
    .pause(1)

  setUp(
    scn.inject(rampUsers(50).during(10.seconds))
  ).protocols(httpProtocol)
   .assertions(
     global.responseTime.percentile3.lt(1000), // p95 < 1s
     global.successfulRequests.percent.gt(95)
   )
}
'''
with open('project/gatling/src/main/scala/ProductsLoadSimulation.scala', 'w') as f:
    f.write(scala_sim)
print("Created ProductsLoadSimulation.scala")
print("Run with: gatling.sh (or mvn gatling:test if using the Gatling Maven plugin)")


Created ProductsLoadSimulation.scala
Run with: gatling.sh (or mvn gatling:test if using the Gatling Maven plugin)


## 4. Analyzing Results / Identifying Bottlenecks
After running either tool you get an HTML report. Key metrics to inspect:

| Metric | What it tells you |
|---|---|
| **Response time (avg / p95 / p99)** | Whether most requests are fast but a tail is slow (points to occasional resource contention) |
| **Throughput (req/s)** | Whether the server saturates under load — a plateau while users increase = a bottleneck |
| **Error rate** | Whether the app starts failing (timeouts, 5xx) under load |
| **Active threads/users vs response time** | Correlating rising latency with rising concurrency pinpoints the load level where the bottleneck appears |

In our sample app, the `time.sleep(random.uniform(0.05, 0.4))` simulates a slow downstream dependency (e.g. a database) — the load test report should show this as the dominant contributor to response time, which is exactly the kind of bottleneck this exercise is meant to surface.